# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. 

Source: Original local package: 02_mit_6s191/04_transformer_contrastive_prototype.ipynb

In [1]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '02_mit_6s191', 'torch', '04_transformer_contrastive_prototype.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: /content/HaiLong_Colab_Exercises/02_mit_6s191
Run mode: FULL EXERCISE
Results: /content/drive/MyDrive/HaiLong_Colab_Results/02_mit_6s191/results


# LO3 extension: transformer and contrastive representation prototype
Original learning-plan extension using synthetic wearable signals. This demonstrates positional encoding, attention, two augmented views, an NT-Xent objective, and a frozen linear probe with subject splits. **It is not SigRep or the Wu et al. transformer implementation.** Paper-specific architecture, augmentation, losses and evaluation must be checked before benchmark claims.


In [2]:
import copy,json
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from wearable_utils import *
torch.manual_seed(42);torch.set_num_threads(1)
x,y,subjects=make_wearable_windows()
train=np.flatnonzero(subjects<5);valid=np.flatnonzero(subjects==5);test=np.flatnonzero(subjects>5)
scaler=fit_signal_scaler(x[train]);xx=torch.tensor(transform_signals(scaler,x))
class SignalTransformer(nn.Module):
    def __init__(self,length=64,width=24):
        super().__init__();self.input=nn.Linear(3,width)
        self.position=nn.Parameter(torch.randn(1,length,width)*.02)
        block=nn.TransformerEncoderLayer(width,4,48,dropout=.1,batch_first=True)
        self.encoder=nn.TransformerEncoder(block,1)
        self.projector=nn.Sequential(nn.Linear(width,width),nn.ReLU(),nn.Linear(width,16))
    def forward(self,x,project=False):
        h=self.encoder(self.input(x)+self.position[:,:x.shape[1]]).mean(1)
        return self.projector(h) if project else h
def augment(x):
    noise=.03*torch.randn_like(x);scale=1+.05*torch.randn(x.shape[0],1,x.shape[-1])
    return x*scale+noise
def nt_xent(a,b,temperature=.2):
    z=nn.functional.normalize(torch.cat([a,b]),dim=1);n=len(a)
    logits=z@z.T/temperature
    logits=logits.masked_fill(torch.eye(2*n,dtype=torch.bool,device=z.device),-1e9)
    positives=(torch.arange(2*n,device=z.device)+n)%(2*n)
    return nn.functional.cross_entropy(logits,positives)
model=SignalTransformer();optimizer=torch.optim.Adam(model.parameters(),lr=.002);history=[]
for epoch in range(6):
    model.train();losses=[]
    for batch in torch.as_tensor(train)[torch.randperm(len(train))].split(32):
        optimizer.zero_grad(set_to_none=True)
        loss=nt_xent(model(augment(xx[batch]),True),model(augment(xx[batch]),True))
        loss.backward();optimizer.step();losses.append(float(loss.detach()))
    history.append({'epoch':epoch,'nt_xent_loss':np.mean(losses)})
model.eval()
with torch.no_grad():representations=model(xx).numpy()
assert representations.shape==(len(x),24)
assert np.isfinite(representations).all()

In [3]:
choices=[]
for c in [.1,1.,10.]:
    probe=make_pipeline(StandardScaler(),LogisticRegression(C=c,class_weight='balanced',max_iter=500))
    probe.fit(representations[train],y[train]);pred=probe.predict(representations[valid])
    choices.append((classification_report_values(y[valid],pred)['macro_f1'],c,probe))
score,c,probe=max(choices,key=lambda item:item[0])
report=classification_report_values(y[test],probe.predict(representations[test]))
results=Path('results');results.mkdir(exist_ok=True)
pd.DataFrame(history).to_csv(results/'synthetic_transformer_pretraining.csv',index=False)
(results/'synthetic_transformer_probe.json').write_text(json.dumps({'C':c,'validation_macro_f1':score,
    'test':report,'dataset':'synthetic','unlabelled_pretraining_subjects':[0,1,2,3,4]},indent=2))
print(report)
print('Held-out subjects were excluded from unsupervised pretraining as well as supervised fitting.')

{'macro_f1': 0.39170314780070875, 'balanced_accuracy': 0.48611111111111116}
Held-out subjects were excluded from unsupervised pretraining as well as supervised fitting.


## Benchmark implementation checklist
Before replacing synthetic data: verify the exact signal channels/rates; labels and excluded classes; participant/session splits; filtering and normalisation; window duration/stride; positive/negative pair construction; augmentations; temperature and loss weighting; pretraining population; downstream label budget; seeds; model selection; metrics; and missing-modality protocol against each paper. The current encoder pools across modalities jointly; it does not reproduce modality-specific attention fusion.


Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [4]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [5]:
# Added by the course runner: inspect the executed runtime, not coursework.
import os as _hailong_os, json as _hailong_json, platform as _hailong_platform
_hailong_evidence = {
    'python': _hailong_platform.python_version(),
    'quick_run': bool(globals().get('QUICK_RUN', False)),
    'course_smoke': _hailong_os.environ.get('COURSE_SMOKE', '0'),
    'keras_backend': _hailong_os.environ.get('KERAS_BACKEND'),
    'legacy_keras': _hailong_os.environ.get('TF_USE_LEGACY_KERAS'),
}
print('__HAILONG_RUNTIME_EVIDENCE__' + _hailong_json.dumps(_hailong_evidence))


__HAILONG_RUNTIME_EVIDENCE__{"python": "3.12.13", "quick_run": false, "course_smoke": "0", "keras_backend": null, "legacy_keras": null}
